In [28]:
import pandas as pd
import json
import random

In [29]:
new_triples_dmd = pd.read_csv('../approved_triples_latest.csv')
print(new_triples_dmd.shape[0])
new_triples_dmd.head()

333


,relation,x_type,y_type,x_id,y_id,uid,x_preferred_name,y_preferred_name,x_preferred_name_score,y_preferred_name_score
0,disease_phenotype_positive,disease,effect/phenotype,kg4rd:10679,kg4rd:1249,:40400011:8cxiHTfHEbuMJ5cudTfhtn,duchenne muscular dystrophy,intellectual disability,1000.0,1000.0
1,anatomy_protein_present,anatomy,gene/protein,kg4rd:15230,kg4rd:1756,:40368879:KSNJHm4UB6yne29TsVe6b2,dorsal vessel heart,dmd,1000.0,1000.0
2,disease_protein,disease,gene/protein,kg4rd:10679,kg4rd:1431,:40366239:52d5FmoktMw9P8H9yRsyEL,duchenne muscular dystrophy,cs,1000.0,1000.0
3,disease_protein,disease,gene/protein,kg4rd:10679,kg4rd:9780,:40361216:QW9DbX3U4yc8iP2ugNLKfx,duchenne muscular dystrophy,piezo1,1000.0,1000.0
4,disease_phenotype_positive,disease,effect/phenotype,kg4rd:10679,kg4rd:3287,:40359883:JrogmgpD3XR4BMnqfQWogW,duchenne muscular dystrophy,abnormality of mitochondrial metabolism,1000.0,1000.0


In [31]:
nodes = pd.read_csv('../../../kg/nodes.csv')
print(nodes.shape[0])
nodes.head()

151321


,node_index,node_id,node_type,node_name,node_source
0,0,kg4rd:381,gene/protein,ARF5,NCBI
1,1,kg4rd:4074,gene/protein,M6PR,NCBI
2,2,kg4rd:2288,gene/protein,FKBP4,NCBI
3,3,kg4rd:56603,gene/protein,CYP26B1,NCBI
4,4,kg4rd:55471,gene/protein,NDUFAF7,NCBI


In [32]:
nodes_d = nodes.drop_duplicates(['node_id', 'node_type'], keep='first')
nodes_d.shape[0]

151112

In [33]:
df = pd.merge(new_triples_dmd, nodes_d, left_on=['x_id', 'x_type'], right_on=['node_id', 'node_type'], how='left').rename(columns={'node_index': 'x_index'}).get(
    ['relation', 'x_index', 'y_id', 'y_type']
).astype({'x_index': int}).astype({'x_index': str})

print(df.shape[0])

df = pd.merge(df, nodes_d, left_on=['y_id', 'y_type'], right_on=['node_id', 'node_type'], how='left').rename(columns={'node_index': 'y_index'}).get(
    ['relation', 'x_index', 'y_index']
).astype({'y_index': int}).astype({'y_index': str})

print(df.shape[0])
df.head()

333
333


,relation,x_index,y_index
0,disease_phenotype_positive,27017,29854
1,anatomy_protein_present,103463,9016
2,disease_protein,27017,7812
3,disease_protein,27017,4785
4,disease_phenotype_positive,27017,32707


In [34]:
edges = pd.read_csv('../../../kg/edges.csv')
print(edges.shape[0])
edges.head()

14641692


,relation,display_relation,x_index,y_index
0,protein_protein,ppi,0,1898
1,protein_protein,ppi,0,769
2,protein_protein,ppi,0,15031
3,protein_protein,ppi,0,2385
4,protein_protein,ppi,0,4981


In [35]:
edges_new = pd.concat([
    edges,
    df
], ignore_index=True)
print(edges_new.shape[0])
edges_new.tail()

14642025


,relation,display_relation,x_index,y_index
14642020,bioprocess_protein,NaN,67142,2241
14642021,bioprocess_protein,NaN,67142,8107
14642022,bioprocess_protein,NaN,67142,13265
14642023,bioprocess_protein,NaN,67142,2884
14642024,bioprocess_protein,NaN,67142,8204


In [37]:
with open('./data/entity2id.txt', 'w') as f:
    f.write(f'{len(nodes)}\n')
    for _, row in nodes.iterrows():
        f.write(f'{row["node_name"]}:{row["node_type"]}\t{row["node_index"]}\n')

In [19]:
relations = edges_new.drop_duplicates(subset=['relation'])['relation'].to_list()
relation2id = {relation: i for i, relation in enumerate(relations)}

with open('./data/relation2id.txt', 'w') as f:
    f.write(f'{len(relations)}\n')
    for k, v in relation2id.items():
        f.write(f'{k}\t{v}\n')

In [38]:
edges_new['rela_id'] = edges_new['relation'].apply(lambda x: relation2id[x])
edges_new = edges_new[['x_index', 'y_index', 'rela_id']]
data = list(edges_new.itertuples(index=False, name=None))
random.shuffle(data)
data[:10]

[(3087, 9305, 0),
 (19934, 28718, 5),
 (3123, 71501, 18),
 (13506, 100976, 28),
 (105125, 10758, 28),
 (10846, 4038, 0),
 (95037, 4054, 28),
 (8764, 13501, 0),
 (118131, 98143, 28),
 (19305, 19886, 5)]

In [39]:
with open('./data/all2id.txt', 'w') as f:
    f.write(f'{len(data)}\n')
    for x, y, r in data:
        f.write(f'{x} {y} {r}\n')

In [40]:
edges['rela_id'] = edges['relation'].apply(lambda x: relation2id[x])
edges = edges[['x_index', 'y_index', 'rela_id']]
data_base = list(edges.itertuples(index=False, name=None))
random.shuffle(data_base)

with open('./data/all2id_base.txt', 'w') as f:
    f.write(f'{len(data_base)}\n')
    for x, y, r in data_base:
        f.write(f'{x} {y} {r}\n')

In [41]:
train_size = int(len(data) * 0.9)
valid_size = int(len(data) * 0.05)
test_size = len(data) - train_size - valid_size

print('train size: ', train_size)
print('valid size: ', valid_size)
print('test size: ', test_size)

train_data = data[:train_size]
valid_data = data[train_size:train_size+valid_size]
test_data = data[train_size+valid_size:]

train size:  13177822
valid size:  732101
test size:  732102


In [43]:
with open('./data/train2id.txt', 'w') as f:
    f.write(f'{len(train_data)}\n')
    for x, y, r in train_data:
        f.write(f'{x} {y} {r}\n')

with open('./data/valid2id.txt', 'w') as f:
    f.write(f'{len(valid_data)}\n')
    for x, y, r in valid_data:
        f.write(f'{x} {y} {r}\n')

with open('./data/test2id.txt', 'w') as f:
    f.write(f'{len(test_data)}\n')
    for x, y, r in test_data:
        f.write(f'{x} {y} {r}\n')